<a href="https://colab.research.google.com/github/Omar-Qaid/CNNs/blob/main/Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.

Found existing installation: unsloth 2026.3.3
Uninstalling unsloth-2026.3.3:
  Successfully uninstalled unsloth-2026.3.3
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-req-build-z4np0uve
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-req-build-z4np0uve
  Resolved https://github.com/unslothai/unsloth.git to commit 69330957eef1dd902bc50603b838f6ed8aad4ff3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.3.3-py3-none-any.whl size=454397 sha256=fb911037eb61ff22d84eed0ccca2b7f1f584bd2a8c6035945d1494aafa71bb5f
  Stored in directory: /tmp/pip-ephem-wheel-cache-trfs2ksr/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [1]:
from datasets import load_dataset

# 1. Define the new prompt template for your specific columns
# We'll focus on the process name, the employee responsible, and the result.
process_prompt = """### Process: {} ({})
Employee: {} ({})
Request ID: {}
Status: Started on {} and finished on {}
Detail: {} - {}

### Summary:
The employee {} handled the "{}" request.
The control value recorded was {} with an actual weight of {}.
"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    # Extracting the new columns from your dataset
    p_names = examples["ProcessName"]
    p_names_ar = examples["ProcessNameAR"]
    emp_names = examples["EmployeeName"]
    emp_codes = examples["EmployeeCode"]
    req_ids = examples["RequestId"]
    start_dates = examples["CreatedDate"]
    end_dates = examples["FinishedDate"]
    ctrl_labels = examples["ControlLabel"]
    ctrl_values = examples["ControlValue"]
    weights = examples["ActualWeight"]

    texts = []
    for i in range(len(p_names)):
        # Format the string using the mapped columns
        text = process_prompt.format(
            p_names[i], p_names_ar[i],
            emp_names[i], emp_codes[i],
            req_ids[i],
            start_dates[i], end_dates[i],
            ctrl_labels[i], ctrl_values[i],
            emp_names[i], p_names[i],
            ctrl_values[i], weights[i]
        ) + EOS_TOKEN
        texts.append(text)

    return { "text": texts }

# 2. Load and Map
# Note: Ensure the path to your dataset is correct (local file or HF path)
dataset = load_dataset("csv", data_files="your_data.csv", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
pass

from datasets import load_dataset
dataset = load_dataset("relai-ai/legal-scenarios-SCOTUS-2024-decisions", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

NameError: name 'tokenizer' is not defined

In [ ]:
dataset['text']

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
try:
    trainer_stats = trainer.train()
    print("Training completed successfully!")
    print(f"Training stats: {trainer_stats}")
except RuntimeError as e:
    print(f"Runtime error occurred: {e}")

In [ ]:
model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")

In [ ]:
model.push_to_hub_gguf("eslamyounis02/law_questions_and_answers", tokenizer, quantization_method = "f16", token = "")